In [ ]:
%env WORKDIR=/tmp/vault

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("/tmp/vault/config.env")

VAULT_TOKEN = os.getenv('VAULT_TOKEN')
VAULT_ADDR = os.getenv('VAULT_ADDR')
VAULT_CACERT = os.getenv('VAULT_CACERT')

# Basic Configuration

In [ ]:
%%bash

vault secrets enable transit
# Create a new key named "kms" in the transit secrets engine
# AES-GCM is the default encryption algorithm used by the transit secrets engine. It is a symmetric encryption algorithm that provides both confidentiality and integrity for the data being encrypted. AES-GCM is widely used in modern cryptography and is considered secure for most applications.

vault write -f transit/keys/kms

# Rotate key every 1 month (30 days) to ensure that the key is regularly updated and to reduce the risk of compromise. 
# By default will will be able to use old keys for decryption, but new encryption operations will use the latest key version. 
# This helps to ensure that sensitive data is protected with the most up-to-date cryptographic algorithms and reduces the risk of data breaches.
vault write transit/keys/kms/config auto_rotate_period=30d


# Encrypt and Decrypt with CLI

In [ ]:
%%bash
# Encrypt a sample plaintext using the "kms" key in the transit secrets engine
PLAINTEXT=$(echo -n "Hello, Vault!" | base64)
CIPHERTEXT=$(vault write transit/encrypt/kms plaintext=$PLAINTEXT -format=json | jq -r '.data.ciphertext')
echo "Ciphertext: $CIPHERTEXT"

# Decrypt the ciphertext using the "kms" key in the transit secrets engine
DECRYPTED=$(vault write transit/decrypt/kms ciphertext=$CIPHERTEXT -format=json | jq -r '.data.plaintext' | base64 --decode)
echo "Decrypted: $DECRYPTED"   


# Encrypt and Decrypt with curl

In [ ]:
%%bash
# Same exercise as above but using curl insetad of the vault cli
# Encrypt a sample plaintext using the "kms" key in the transit secrets engine
PLAINTEXT=$(echo -n "Hello, Vault!" | base64)
curl -k -s --header "X-Vault-Token: $VAULT_TOKEN" \
     --request POST \
     --data '{"plaintext": "'"$PLAINTEXT"'"}' \
     $VAULT_ADDR/v1/transit/encrypt/kms | jq -r '.data.ciphertext'  > ciphertext.txt
CIPHERTEXT=$(cat ciphertext.txt)
echo "Ciphertext: $CIPHERTEXT"

# Decrypt the ciphertext using the "kms" key in the transit secrets engine
DECRYPTED=$(curl -k -s --header "X-Vault-Token: $VAULT_TOKEN" \
     --request POST \
     --data '{"ciphertext": "'"$CIPHERTEXT"'"}' \
     $VAULT_ADDR/v1/transit/decrypt/kms | jq -r '.data.plaintext' | base64 --decode)
echo "Decrypted: $DECRYPTED"


# Batch operations

In [ ]:
%%bash
# Batch - Generate 1000 inputs for batch encryption

echo "Generating batch_input.json with 1000 unique card entries..."

# Start building JSON array
echo '{' > batch_input.json
echo '"batch_input": [' >> batch_input.json

# Generate 1000 entries, each with a unique card number derived from the index
for i in {1..1000}; do
    # Build a unique 16-digit card number: Visa prefix (4) + zero-padded index
    num=$(printf "4%015d" $i)
    card_number="${num:0:4} ${num:4:4} ${num:8:4} ${num:12:4}"

    # Base64 encode the plaintext
    encoded_text=$(echo -n "$card_number" | base64)

    # Add JSON entry (no trailing comma on the last entry)
    if [ $i -lt 1000 ]; then
        echo "    {\"plaintext\": \"$encoded_text\"}," >> batch_input.json
    else
        echo "    {\"plaintext\": \"$encoded_text\"}" >> batch_input.json
    fi

    # Progress indicator
    if [ $((i % 100)) -eq 0 ]; then
        echo "Generated $i entries..."
    fi
done

# Close JSON array
echo ']' >> batch_input.json
echo '}' >> batch_input.json

echo "Generated batch_input.json with 1000 unique entries"
echo "File size: $(wc -c < batch_input.json) bytes"
echo "Total lines: $(wc -l < batch_input.json) lines"

echo -e "\nFirst 10 lines of the file:"
head -10 batch_input.json

echo -e "\nLast 10 lines of the file:"
tail -10 batch_input.json


## Encrypt

In [ ]:
%%bash
# Execute batch encryption with 1000 inputs

echo "Executing batch encryption of 1000 entries..."
echo "Start time: $(date)"

# Execute the batch operation and measure time
start_time=$(date +%s%N)

curl -k -s \
    --header "X-Vault-Token: $VAULT_TOKEN" \
    --request POST \
    --data @batch_input.json \
    $VAULT_ADDR/v1/transit/encrypt/orders > batch_output.json

end_time=$(date +%s%N)
duration_ns=$((end_time - start_time))
duration_seconds=$(echo "scale=6; $duration_ns / 1000000000" | bc)

echo "Batch encryption completed!"
echo "Duration: ${duration_ns} nanoseconds"
echo "Duration: ${duration_seconds} seconds"

# Calculate operations per second (avoid division by zero)
if [ $duration_ns -gt 0 ]; then
    # Use bc for floating point arithmetic: (1000 * 1000000000) / duration_ns
    ops_per_second=$(echo "scale=2; 1000000000000 / $duration_ns" | bc)
    echo "Processing rate: ${ops_per_second} operations/second"
else
    echo "Processing rate: Unable to calculate (duration too small)"
fi

echo -e "\nOutput file size: $(wc -c < batch_output.json) bytes"

# Check if the operation was successful
if jq -e '.data.batch_results' batch_output.json > /dev/null 2>&1; then
    result_count=$(jq '.data.batch_results | length' batch_output.json)
    echo "Successfully encrypted $result_count entries"
    
    echo -e "\nFirst encrypted result:"
    jq '.data.batch_results[0]' batch_output.json
    
    echo -e "\nLast encrypted result:"
    jq '.data.batch_results[-1]' batch_output.json
else
    echo "Batch operation failed. Response:"
    jq '.' batch_output.json
fi


## Decrypt

### Adapt input batch file for decrypt operation

In [ ]:
%%bash
# Extract encrypted results and format for batch decryption

echo "Processing batch_output.json for decryption..."

# Use jq to build a properly formatted batch_decrypt_input.json directly
jq '{batch_input: [.data.batch_results[] | {ciphertext: .ciphertext}]}' \
    batch_output.json > batch_decrypt_input.json

echo "Created batch_decrypt_input.json for decryption"
echo "File size: $(wc -c < batch_decrypt_input.json) bytes"

echo -e "\nFirst 5 lines of decrypt input:"
head -5 batch_decrypt_input.json

echo -e "\nValidating JSON format:"
if jq empty batch_decrypt_input.json 2>/dev/null; then
    echo "JSON format is valid"
    entry_count=$(jq '.batch_input | length' batch_decrypt_input.json)
    echo "Total entries to decrypt: $entry_count"
else
    echo "JSON format is invalid"
fi


### Decrypt

In [ ]:
%%bash
# Execute batch decryption with 1000 entries

echo "Executing batch decryption of 1000 entries..."
echo "Start time: $(date)"

# Execute the batch decryption operation and measure time
start_time=$(date +%s%N)

curl -k -s \
    --header "X-Vault-Token: $VAULT_TOKEN" \
    --request POST \
    --data @batch_decrypt_input.json \
    $VAULT_ADDR/v1/transit/decrypt/orders > batch_decrypt_output.json

end_time=$(date +%s%N)
duration_ns=$((end_time - start_time))
duration_seconds=$(echo "scale=6; $duration_ns / 1000000000" | bc)

echo "Batch decryption completed!"
echo "Duration: ${duration_ns} nanoseconds"
echo "Duration: ${duration_seconds} seconds"

# Calculate operations per second (avoid division by zero)
if [ $duration_ns -gt 0 ]; then
    # Use bc for floating point arithmetic: (1000 * 1000000000) / duration_ns
    ops_per_second=$(echo "scale=2; 1000000000000 / $duration_ns" | bc)
    echo "Processing rate: ${ops_per_second} operations/second"
else
    echo "Processing rate: Unable to calculate (duration too small)"
fi

echo -e "\nOutput file size: $(wc -c < batch_decrypt_output.json) bytes"

# Check if the operation was successful
if jq -e '.data.batch_results' batch_decrypt_output.json > /dev/null 2>&1; then
    result_count=$(jq '.data.batch_results | length' batch_decrypt_output.json)
    echo "Successfully decrypted $result_count entries"
    
    echo -e "\nFirst decrypted result (base64 encoded):"
    jq '.data.batch_results[0]' batch_decrypt_output.json
    
    echo -e "\nFirst decrypted result (decoded):"
    jq -r '.data.batch_results[0].plaintext' batch_decrypt_output.json | base64 -d
    
    echo -e "\nLast decrypted result (decoded):"
    jq -r '.data.batch_results[-1].plaintext' batch_decrypt_output.json | base64 -d
    
    # Show a few more samples
    echo -e "\nSample of decrypted entries (first 5):"
    for i in {0..4}; do
        decrypted=$(jq -r ".data.batch_results[$i].plaintext" batch_decrypt_output.json | base64 -d)
        echo "Entry $((i+1)): $decrypted"
    done
else
    echo "Batch decryption failed. Response:"
    jq '.' batch_decrypt_output.json
fi

# Rotate key

In [ ]:
%%bash
# Rotate the key to demonstrate decryption of old ciphertexts with a new key version
echo "Rotating the 'kms' key to create a new version..."    

vault write -f transit/keys/kms/rotate 

In [ ]:
%%bash
# Encrypt a sample plaintext using the "kms" key in the transit secrets engine
PLAINTEXT=$(echo -n "Hello, Vault!" | base64)
CIPHERTEXT=$(vault write transit/encrypt/kms plaintext=$PLAINTEXT -format=json | jq -r '.data.ciphertext')
echo "Ciphertext: $CIPHERTEXT"

## Decrypt data with old and new keys

In [ ]:
%%bash
# Encrypt using the new key version to demonstrate that it can still be decrypted after rotation
PLAINTEXT=$(echo -n "Hello, Vault! (New Key Version)" | base64)
CIPHERTEXT_NEW=$(vault write transit/encrypt/kms plaintext=$PLAINTEXT -format=json | jq -r '.data.ciphertext')
echo "Ciphertext (New Key Version): $CIPHERTEXT_NEW"

# Decrypt the new ciphertext using the "kms" key in the transit secrets engine
DECRYPTED_NEW=$(vault write transit/decrypt/kms ciphertext=$CIPHERTEXT_NEW -format=json | jq -r '.data.plaintext' | base64 --decode)
echo "Decrypted (New Key Version): $DECRYPTED_NEW" 


In [ ]:
%%bash
CYPHERTEXT_OLD=$(cat ciphertext.txt)
echo "Old Ciphertext: $CYPHERTEXT_OLD"
# Decrypt the old ciphertext using the "kms" key in the transit secrets engine
DECRYPTED_OLD=$(vault write transit/decrypt/kms ciphertext=$CYPHERTEXT_OLD -format=json | jq -r '.data.plaintext' | base64 --decode)
echo "Decrypted (Old Key Version): $DECRYPTED_OLD"

In [ ]:
%%bash
# Rewrap the old ciphertext to the new key version
CYPHERTEXT_OLD=$(cat ciphertext.txt)
echo "Old Ciphertext: $CYPHERTEXT_OLD"

# Rewrap the old ciphertext to the new key version using the "kms" key in the transit secrets engine
REWRAPPED_CIPHERTEXT=$(vault write transit/rewrap/kms ciphertext=$CYPHERTEXT_OLD -format=json | jq -r '.data.ciphertext')
echo "Rewrapped Ciphertext (New Key Version): $REWRAPPED_CIPHERTEXT"    

# Decrypt the rewrapped ciphertext using the "kms" key in the transit secrets engine
DECRYPTED_REWRAPPED=$(vault write transit/decrypt/kms ciphertext=$REWRAPPED_CIPHERTEXT -format=json | jq -r '.data.plaintext' | base64 --decode)
echo "Decrypted (Rewrapped to New Key Version): $DECRYPTED_REWRAPPED"